In [12]:
import random
number = random.randint(0, 119)
up_down = random.randint(0, 1)
up_down_list = ["up", "down"]
#print(up_down_list)
#print(up_down)
print(number, up_down_list[up_down])

31 down


In [37]:
import numpy as np

In [13]:
x = 1993
x - 1941 

52

In [ ]:
x = 1982
x - 1941 

In [38]:
np.log(2)

np.float64(0.6931471805599453)

In [69]:
2**1.6931471805599453

3.2336133444833495

In [35]:
2.3025**np.e

9.650741802925653

In [158]:
x = -3
round(1/(1+ np.e**x), 2)

0.95

In [ ]:
import numpy as np

ModuleNotFoundError: No module named 'numpy'

In [57]:
def logit_math(x):
    x = x/2
    return round(1/(1+ np.e**x), 2)

In [25]:
for i in list(range(-10, 10, 1)):
    print(logit_math(i))

NameError: name 'logit_math' is not defined

In [110]:
90_000 * 25

2250000

In [111]:
90_000 * 35

3150000

In [ ]:
13 million albums

In [114]:
won_2_usd = 1449.83
album_revenue =  (55_000_000_000 / won_2_usd)/ 1_000_000
total_revenue = (171_000_000_000 / won_2_usd) / 1_000_000
print("Album Revenue", album_revenue)
print("Total Revenue",total_revenue)

Album Revenue 37.93548209100378
Total Revenue 117.94486250112082


In [115]:
total_revenue - album_revenue

80.00938041011705

In [117]:
album_revenue /2

18.96774104550189

In [ ]:
$38 Million in Physical Album Sales in Quarter 3 of 2024



In [3]:
import numpy as np
import pandas as pd
import pymc3 as pm
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Define the venues and their entrances
venues = {
    'Mercedes_Benz_Stadium': {
        'entrances': ['Gate_A', 'Gate_B', 'Gate_C', 'Gate_D', 'Gate_E'],
        'total_estimate': 77462,  # Your point estimate
        'uncertainty': 0.1,       # 10% uncertainty
        'eligible_percent': 1.0   # All attendees get a cow
    },
    'GWCC': {
        'entrances': ['Main_Entrance', 'Exhibit_Hall_A', 'Exhibit_Hall_B', 'North_Entrance', 'South_Entrance'],
        'total_estimate': 6186,   # Your point estimate for children under 10
        'uncertainty': 0.1,       # 10% uncertainty
        'eligible_percent': 0.14  # Estimated percentage of attendees who are children under 10
    }
}

# Create a timestamp range for the events
# Championship game: January 13, 2025 (second Monday) from 6pm to 10pm (gates open 2 hours before)
game_date = datetime(2025, 1, 13)
game_hours = pd.date_range(
    start=game_date.replace(hour=18, minute=0),  # 6pm
    end=game_date.replace(hour=22, minute=0),    # 10pm
    freq='1H'
)

# Kickoff Experience: January 11-13, 2025 (Sat-Mon) from 10am to 6pm
kickoff_hours = []
for day in range(11, 14):  # Jan 11, 12, 13
    kickoff_hours.extend(
        pd.date_range(
            start=datetime(2025, 1, day, 10, 0),  # 10am
            end=datetime(2025, 1, day, 18, 0),    # 6pm
            freq='1H'
        )
    )
kickoff_hours = pd.DatetimeIndex(kickoff_hours)

# Expected attendance patterns (normalized) - these would be based on historical data
# For the championship game, heaviest influx is expected 1-2 hours before game time
mbs_hourly_pattern = np.array([0.25, 0.40, 0.25, 0.10])  # 6pm, 7pm, 8pm, 9pm

# For the Kickoff Experience, distribution varies by day and hour
# These patterns are hypothetical and should be replaced with actual data if available
kickoff_daily_pattern = np.array([0.3, 0.4, 0.3])  # Sat, Sun, Mon distribution
kickoff_hourly_pattern = np.array([0.05, 0.10, 0.15, 0.20, 0.20, 0.15, 0.10, 0.05])  # 10am to 6pm

# Function to generate synthetic prior data based on our estimates
def generate_prior_distributions(venues):
    priors = {}
    
    for venue_name, venue_info in venues.items():
        total_estimate = venue_info['total_estimate']
        uncertainty = venue_info['uncertainty']
        
        # Create a normal distribution for the prior
        # Mean is our best estimate, standard deviation reflects our uncertainty
        std_dev = total_estimate * uncertainty
        
        if venue_name == 'Mercedes_Benz_Stadium':
            # Stadium has specific hours
            hours = game_hours
            hourly_pattern = mbs_hourly_pattern
            # Expected distribution across entrances (could be derived from historical data)
            entrance_weights = np.array([0.25, 0.20, 0.20, 0.20, 0.15])
        else:
            # GWCC Kickoff Experience
            hours = kickoff_hours
            # Need to expand daily pattern to hourly
            hourly_pattern = np.zeros(len(kickoff_hours))
            
            hour_idx = 0
            for day_idx, day_weight in enumerate(kickoff_daily_pattern):
                for hour_weight in kickoff_hourly_pattern:
                    hourly_pattern[hour_idx] = day_weight * hour_weight
                    hour_idx += 1
            
            # Expected distribution across entrances
            entrance_weights = np.array([0.30, 0.20, 0.20, 0.15, 0.15])
        
        # Normalize to ensure sum = 1
        hourly_pattern = hourly_pattern / hourly_pattern.sum()
        entrance_weights = entrance_weights / entrance_weights.sum()
        
        # Allocate expected attendees by hour
        hourly_attendees = hourly_pattern * total_estimate
        
        # For each hour, allocate expected attendees by entrance
        entrance_hourly_attendees = {}
        for i, entrance in enumerate(venue_info['entrances']):
            entrance_hourly_attendees[entrance] = hourly_attendees * entrance_weights[i]
        
        priors[venue_name] = {
            'mean': total_estimate,
            'std_dev': std_dev,
            'hours': hours,
            'hourly_pattern': hourly_pattern,
            'entrance_weights': entrance_weights,
            'entrance_hourly_attendees': entrance_hourly_attendees
        }
    
    return priors

# Generate prior distributions
prior_distributions = generate_prior_distributions(venues)

# Bayesian update function for a single entrance at a specific hour
def bayesian_update(prior_mean, prior_std, observed_count, observation_std):
    """
    Update a prior distribution with new observations using Bayesian inference
    
    Parameters:
    prior_mean (float): Prior mean value
    prior_std (float): Prior standard deviation
    observed_count (float): Observed count
    observation_std (float): Standard deviation of observation (measurement uncertainty)
    
    Returns:
    tuple: (posterior_mean, posterior_std)
    """
    # Precision (inverse variance) of prior and likelihood
    prior_precision = 1 / (prior_std ** 2)
    likelihood_precision = 1 / (observation_std ** 2)
    
    # Posterior precision
    posterior_precision = prior_precision + likelihood_precision
    
    # Posterior mean
    posterior_mean = (prior_mean * prior_precision + observed_count * likelihood_precision) / posterior_precision
    
    # Posterior standard deviation
    posterior_std = np.sqrt(1 / posterior_precision)
    
    return posterior_mean, posterior_std

# Simulate real-time updating of forecasts with mock data
def simulate_real_time_forecasting(venues, prior_distributions, simulation_hours=None, noise_level=0.2):
    """
    Simulate real-time forecasting with mock data
    
    Parameters:
    venues (dict): Venue configuration
    prior_distributions (dict): Prior distribution data
    simulation_hours (int): Number of hours to simulate (None for full event)
    noise_level (float): Level of random noise to add to simulated observations
    
    Returns:
    dict: Updated forecasts and actual observations
    """
    results = {}
    
    for venue_name, venue_info in venues.items():
        venue_results = {
            'forecast_history': [],
            'observations': [],
            'entrances': {}
        }
        
        prior = prior_distributions[venue_name]
        hours = prior['hours']
        
        # Limit simulation hours if specified
        if simulation_hours is not None:
            hours = hours[:simulation_hours]
        
        # Initialize running totals
        total_observed = 0
        total_forecast = prior['mean']
        forecast_std = prior['std_dev']
        
        # Track by entrance
        for entrance in venue_info['entrances']:
            venue_results['entrances'][entrance] = {
                'forecast_history': [],
                'observations': []
            }
        
        # Process each hour
        for hour_idx, hour in enumerate(hours):
            hour_forecast = {}
            hour_observed = {}
            
            # For each entrance
            for entrance in venue_info['entrances']:
                # Get expected attendance for this entrance and hour
                expected_count = prior['entrance_hourly_attendees'][entrance][hour_idx]
                
                # Add some random noise to create "actual" observations
                # This simulates real-world variance
                random_factor = 1 + np.random.normal(0, noise_level)
                actual_count = max(0, expected_count * random_factor)
                
                # Assume some observation uncertainty (e.g., from counting methods)
                observation_std = actual_count * 0.05  # 5% measurement uncertainty
                
                # Get existing forecast for this entrance
                if hour_idx == 0:
                    # First hour uses the prior
                    entrance_forecast_mean = expected_count
                    entrance_forecast_std = expected_count * venue_info['uncertainty']
                else:
                    # Subsequent hours use the previous forecast
                    entrance_data = venue_results['entrances'][entrance]
                    entrance_forecast_mean = entrance_data['forecast_history'][-1]['forecast_mean']
                    entrance_forecast_std = entrance_data['forecast_history'][-1]['forecast_std']
                
                # Update forecast with Bayesian inference
                posterior_mean, posterior_std = bayesian_update(
                    entrance_forecast_mean,
                    entrance_forecast_std,
                    actual_count,
                    observation_std
                )
                
                # Store results
                hour_forecast[entrance] = {
                    'forecast_mean': posterior_mean,
                    'forecast_std': posterior_std,
                    'time': hour
                }
                
                hour_observed[entrance] = {
                    'actual_count': actual_count,
                    'time': hour
                }
                
                # Update entrance history
                venue_results['entrances'][entrance]['forecast_history'].append(hour_forecast[entrance])
                venue_results['entrances'][entrance]['observations'].append(hour_observed[entrance])
                
                # Update totals
                total_observed += actual_count
            
            # Update venue-level forecast using aggregated data
            # In a real implementation, this would be more sophisticated
            venue_forecast = {
                'forecast_mean': total_forecast,
                'forecast_std': forecast_std,
                'time': hour
            }
            
            venue_observation = {
                'total_observed': total_observed,
                'time': hour
            }
            
            venue_results['forecast_history'].append(venue_forecast)
            venue_results['observations'].append(venue_observation)
        
        results[venue_name] = venue_results
    
    return results

# Simulate real-time forecasting for the first few hours of each event
simulation_results = simulate_real_time_forecasting(venues, prior_distributions, simulation_hours=5)

# Function to recommend inventory rebalancing
def recommend_rebalancing(venue_name, current_hour, simulation_results, venues):
    """
    Recommend inventory rebalancing between entrances
    
    Parameters:
    venue_name (str): Name of the venue
    current_hour (int): Current hour of operation
    simulation_results (dict): Simulation results
    venues (dict): Venue configuration
    
    Returns:
    dict: Rebalancing recommendations
    """
    venue_results = simulation_results[venue_name]
    entrances = venues[venue_name]['entrances']
    
    # Get current forecast for each entrance
    current_forecasts = {}
    current_inventory = {}  # This would come from actual inventory tracking system
    
    for entrance in entrances:
        entrance_data = venue_results['entrances'][entrance]
        current_forecast = entrance_data['forecast_history'][current_hour]['forecast_mean']
        
        # In a real system, current inventory would be tracked
        # Here we'll simulate it as initial allocation minus observed
        observed_so_far = sum(obs['actual_count'] for obs in entrance_data['observations'][:current_hour+1])
        initial_allocation = sum(entrance_data['forecast_history'][0]['forecast_mean'] for _ in range(len(entrance_data['observations'])))
        
        # Simulate remaining inventory
        remaining_inventory = max(0, initial_allocation - observed_so_far)
        
        current_forecasts[entrance] = current_forecast
        current_inventory[entrance] = remaining_inventory
    
    # Calculate remaining demand
    remaining_hours = len(venue_results['entrances'][entrances[0]]['forecast_history']) - current_hour - 1
    
    if remaining_hours <= 0:
        return {"message": "Event is complete, no rebalancing needed."}
    
    remaining_demand = {}
    
    for entrance in entrances:
        entrance_data = venue_results['entrances'][entrance]
        future_forecasts = [data['forecast_mean'] for data in entrance_data['forecast_history'][current_hour+1:]]
        remaining_demand[entrance] = sum(future_forecasts)
    
    # Check if any entrance is projected to have a shortage or excess
    shortages = {}
    excesses = {}
    
    for entrance in entrances:
        balance = current_inventory[entrance] - remaining_demand[entrance]
        
        if balance < 0:
            shortages[entrance] = abs(balance)
        elif balance > 50:  # Allow some buffer
            excesses[entrance] = balance
    
    # Recommend rebalancing
    recommendations = []
    
    # Sort entrances by shortage (most critical first)
    shortage_entrances = sorted(shortages.keys(), key=lambda x: shortages[x], reverse=True)
    excess_entrances = sorted(excesses.keys(), key=lambda x: excesses[x], reverse=True)
    
    for shortage_entrance in shortage_entrances:
        shortage_amount = shortages[shortage_entrance]
        
        for excess_entrance in excess_entrances:
            excess_amount = excesses[excess_entrance]
            
            if excess_amount <= 0:
                continue
            
            transfer_amount = min(shortage_amount, excess_amount)
            
            if transfer_amount > 10:  # Only recommend significant transfers
                recommendations.append({
                    'from_entrance': excess_entrance,
                    'to_entrance': shortage_entrance,
                    'amount': int(transfer_amount)
                })
                
                # Update remaining amounts
                shortages[shortage_entrance] -= transfer_amount
                excesses[excess_entrance] -= transfer_amount
    
    return {
        'current_hour': current_hour,
        'venue': venue_name,
        'recommendations': recommendations
    }

# Example: Get rebalancing recommendations for hour 3 of MBS
rebalancing_example = recommend_rebalancing('Mercedes_Benz_Stadium', 3, simulation_results, venues)

# Function to visualize forecasts and actuals
def visualize_forecasts(venue_name, simulation_results):
    """
    Visualize forecasts and actual observations for a venue
    
    Parameters:
    venue_name (str): Name of the venue
    simulation_results (dict): Simulation results
    """
    venue_results = simulation_results[venue_name]
    
    # Extract data for plotting
    times = [data['time'] for data in venue_results['forecast_history']]
    forecast_means = [data['forecast_mean'] for data in venue_results['forecast_history']]
    forecast_stds = [data['forecast_std'] for data in venue_results['forecast_history']]
    observed = [data['total_observed'] for data in venue_results['observations']]
    
    # Create confidence intervals
    upper_bound = [mean + 1.96 * std for mean, std in zip(forecast_means, forecast_stds)]
    lower_bound = [mean - 1.96 * std for mean, std in zip(forecast_means, forecast_stds)]
    
    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(times, forecast_means, 'b-', label='Forecast')
    plt.fill_between(times, lower_bound, upper_bound, color='b', alpha=0.2, label='95% CI')
    plt.plot(times, observed, 'r--', label='Observed')
    plt.title(f'Bayesian Forecast vs. Actual - {venue_name}')
    plt.xlabel('Time')
    plt.ylabel('Cumulative Attendees')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    return plt

# Function to create a dashboard-like summary
def generate_dashboard_summary(simulation_results, venues, current_hour=4):
    """
    Generate a summary of the current state of the distribution
    
    Parameters:
    simulation_results (dict): Simulation results
    venues (dict): Venue configuration
    current_hour (int): Current hour of the simulation
    
    Returns:
    dict: Dashboard summary data
    """
    dashboard = {}
    
    for venue_name, venue_info in venues.items():
        venue_results = simulation_results[venue_name]
        entrances = venue_info['entrances']
        
        # Extract forecast vs. actual at the venue level
        venue_forecast = venue_results['forecast_history'][current_hour]['forecast_mean']
        venue_observed = venue_results['observations'][current_hour]['total_observed']
        forecast_accuracy = (venue_observed / venue_forecast) * 100 if venue_forecast > 0 else 0
        
        # Get entrance-level data
        entrance_data = []
        
        for entrance in entrances:
            entrance_results = venue_results['entrances'][entrance]
            entrance_forecast = entrance_results['forecast_history'][current_hour]['forecast_mean']
            entrance_observed = entrance_results['observations'][current_hour]['actual_count']
            
            # In a real system, inventory would be tracked
            # Here we simulate it based on initial allocation and observations
            initial_allocation = sum(entrance_results['forecast_history'][0]['forecast_mean'] for _ in range(len(entrance_results['observations'])))
            observed_so_far = sum(obs['actual_count'] for obs in entrance_results['observations'][:current_hour+1])
            remaining_inventory = max(0, initial_allocation - observed_so_far)
            
            entrance_data.append({
                'entrance': entrance,
                'forecast': entrance_forecast,
                'observed': entrance_observed,
                'accuracy': (entrance_observed / entrance_forecast) * 100 if entrance_forecast > 0 else 0,
                'remaining_inventory': remaining_inventory
            })
        
        # Get rebalancing recommendations
        rebalancing = recommend_rebalancing(venue_name, current_hour, simulation_results, venues)
        
        dashboard[venue_name] = {
            'current_hour': current_hour,
            'total_forecast': venue_forecast,
            'total_observed': venue_observed,
            'forecast_accuracy': forecast_accuracy,
            'entrance_data': entrance_data,
            'rebalancing_recommendations': rebalancing.get('recommendations', [])
        }
    
    return dashboard

# Generate a dashboard summary for hour 4
dashboard_summary = generate_dashboard_summary(simulation_results, venues, current_hour=4)

# Example of how to print the dashboard summary (this would be displayed on a real dashboard)
def print_dashboard_summary(dashboard_summary):
    """Print a simplified version of the dashboard summary"""
    for venue_name, venue_data in dashboard_summary.items():
        print(f"\n===== {venue_name} Dashboard =====")
        print(f"Current Hour: {venue_data['current_hour']}")
        print(f"Total Forecast: {venue_data['total_forecast']:.2f}")
        print(f"Total Observed: {venue_data['total_observed']:.2f}")
        print(f"Forecast Accuracy: {venue_data['forecast_accuracy']:.1f}%")
        
        print("\nEntrance Data:")
        for entrance in venue_data['entrance_data']:
            print(f"  {entrance['entrance']}: Forecast={entrance['forecast']:.2f}, "
                 f"Observed={entrance['observed']:.2f}, "
                 f"Remaining Inventory={entrance['remaining_inventory']:.2f}")
        
        print("\nRebalancing Recommendations:")
        if venue_data['rebalancing_recommendations']:
            for rec in venue_data['rebalancing_recommendations']:
                print(f"  Move {rec['amount']} cows from {rec['from_entrance']} to {rec['to_entrance']}")
        else:
            print("  No rebalancing needed at this time.")

# Integration with a real-time system would involve:
# 1. Setting up data collection points at each entrance
# 2. Creating a central database to store observations
# 3. Implementing API endpoints to update the model and get recommendations
# 4. Building a real-time dashboard for operations staff

# Example of what a real-time update function might look like
def update_with_new_observation(venue_name, entrance_name, hour_idx, observed_count, simulation_results):
    """
    Update forecasts with a new observation
    
    Parameters:
    venue_name (str): Name of the venue
    entrance_name (str): Name of the entrance
    hour_idx (int): Hour index
    observed_count (float): Observed count
    simulation_results (dict): Current simulation results
    
    Returns:
    dict: Updated simulation results
    """
    venue_results = simulation_results[venue_name]
    entrance_results = venue_results['entrances'][entrance_name]
    
    # Get current forecast
    if hour_idx == 0:
        # First hour
        prior_mean = entrance_results['forecast_history'][0]['forecast_mean']
        prior_std = entrance_results['forecast_history'][0]['forecast_std']
    else:
        # Use previous forecast
        prior_mean = entrance_results['forecast_history'][hour_idx-1]['forecast_mean']
        prior_std = entrance_results['forecast_history'][hour_idx-1]['forecast_std']
    
    # Observation uncertainty (would be determined by your measurement methods)
    observation_std = observed_count * 0.05  # 5% uncertainty
    
    # Update forecast
    posterior_mean, posterior_std = bayesian_update(
        prior_mean,
        prior_std,
        observed_count,
        observation_std
    )
    
    # Update the simulation results
    entrance_results['forecast_history'][hour_idx]['forecast_mean'] = posterior_mean
    entrance_results['forecast_history'][hour_idx]['forecast_std'] = posterior_std
    entrance_results['observations'][hour_idx]['actual_count'] = observed_count
    
    # In a real system, you would also update venue-level aggregates
    
    return simulation_results

# Main execution function that would run in a real-time system
def main():
    """
    Example of how this would be used in a real-time system
    """
    # 1. Initialize with prior distributions
    venues_config = venues
    prior_distributions = generate_prior_distributions(venues_config)
    
    # 2. Set up initial simulation results (placeholder for real-time data)
    simulation_results = simulate_real_time_forecasting(venues_config, prior_distributions, simulation_hours=0)
    
    # 3. In a real system, this would be a continuous loop with real observations
    # For demonstration, we'll just do a few updates
    for hour_idx in range(5):
        print(f"\n==== Processing Hour {hour_idx} ====")
        
        # In a real system, these would come from entrance counters
        for venue_name, venue_info in venues_config.items():
            for entrance in venue_info['entrances']:
                # Simulate an observation (in real-life this would come from data collection)
                # For demo purposes, we'll generate random observations
                expected = prior_distributions[venue_name]['entrance_hourly_attendees'][entrance][hour_idx]
                observed = expected * (1 + np.random.normal(0, 0.2))  # Add noise
                
                # Update the model
                simulation_results = update_with_new_observation(
                    venue_name, entrance, hour_idx, observed, simulation_results
                )
        
        # 4. Generate dashboard and recommendations
        dashboard = generate_dashboard_summary(simulation_results, venues_config, current_hour=hour_idx)
        print_dashboard_summary(dashboard)
        
        # 5. In a real system, send recommendations to operations team
        # For demo, we'll just print them
        for venue_name in venues_config:
            rebalancing = recommend_rebalancing(venue_name, hour_idx, simulation_results, venues_config)
            if rebalancing.get('recommendations'):
                print(f"\nREBALANCING NEEDED for {venue_name}:")
                for rec in rebalancing['recommendations']:
                    print(f"  Move {rec['amount']} cows from {rec['from_entrance']} to {rec['to_entrance']}")

# Uncomment to run the main simulation
# main()

# For a real implementation, you would need to:
# 1. Replace the synthetic data with actual data collection
# 2. Implement a proper database to store observations and forecasts
# 3. Create API endpoints for data input and recommendation output
# 4. Build a user interface for operations staff
# 5. Set up alerting for critical situations (e.g., imminent shortages)
# 6. Implement more sophisticated inventory tracking

ModuleNotFoundError: No module named 'distutils.msvccompiler'

In [2]:
pip install pymc3

  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached matplotlib-3.10.1-cp310-cp310-macosx_10_12_x86_64.whl.metadata (11 kB)
INFO: pip is looking at multiple versions of arviz to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of arviz to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
  Using cached wrapt-1.17.2-cp310-cp310-macosx_10_9_x86_64.whl.metadata (6.4 kB)
  Using cached contourpy-1.3.1-cp310-cp310-macosx_10_9_x86_64.wh